In [ ]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import date, datetime, timedelta
from itertools import product
from copy import deepcopy
from gen_variable_standard_static import metrics_search_for_two_fragments_df
from tqdm import tqdm

In [ ]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records', 'sample_year'],
      dtype='str')

In [ ]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [ ]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

In [ ]:
def daily_delta_t_spread(
    input_dir: str,
    system_id: int
):
    '''Find all daily changes for all sub-systems.
    
    Parameters:
    ------------
    input_dir: str
        The path to the results of ac_power_parquet_distiller_yearly.py
        Default is data_ds_project/testing_yearly_parquet, 
        but this may change in the final build.
    system_id: int
        The desired system_id.  
    
    Returns:
    ---------
    output_df: pd.DataFrame or None
        If no (good) data, return None
        Else, return a DataFrame with 1-2 columns (from 'inverter', 'meter', 'unknown')
        and indices corresponding to each day with significant data
        (with int data for each of year/month/day) 
    '''
    # initialize data frame
    # know all data is in 1994 - 2023 range
    num_years = 2023-1994 + 1
    num_days = num_years * 12 * 31
    output_df = pd.DataFrame(
        data = np.full((num_days, 6), fill_value=pd.NA),
        index = pd.MultiIndex.from_product(
            [range(1994, 2024), range(1, 13), range(1, 32)]
        ),
        columns = ['inv_downward', 'inv_upward', 'met_downward', 'met_upward', 'unk_downward', 'unk_upward'],
        dtype='object'
    )
    output_df.index.names = ['year', 'month', 'day']
    # set target
    if input_dir[-1] == '/':
        test_folder_str = f'{input_dir}{system_id}/'
    else:
        test_folder_str = f'{input_dir}/{system_id}/'
    test_folder = Path(test_folder_str)
    # columns invariant among years accessed, so can just grab a year with no data to grab column names
    empty_year_pq = pq.ParquetDataset(
        test_folder,
        filters=[('year', '==', 1990)]
    )
    empty_year_df = empty_year_pq.read().to_pandas()

    # grab column names
    col_names = empty_year_df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    for year in range(1994, 2024):
        current_year_pq = pq.ParquetDataset(
            test_folder,
            filters=[
                ('year', '==', year)
            ]            
        )
        current_year_df = current_year_pq.read().to_pandas()
        for pow_col_name in pow_col_names:
            pow_selection = current_year_df[['time', pow_col_name]]
            # drop if no data in the column
            pow_selection = pow_selection.dropna(axis=1, how='all')
            if (pow_selection.shape[1] > 1) and (pow_selection.shape[0] > 10):
                pow_selection.loc[:, 'month'] = pow_selection['time'].dt.month
                pow_selection.loc[:, 'day'] = pow_selection['time'].dt.day
                for month in sorted(pow_selection['month'].unique()):
                    month_selection = pow_selection[pow_selection['month'] == month]
                    clean_month = int(month)
                    if (month_selection.shape[0] > 10):
                        for day in sorted(month_selection['day'].unique()):
                            day_selection = month_selection[month_selection['day'] == day]
                            clean_day = int(day)
                            if day_selection.shape[0] > 2:  # need any nonzero data, I'm afraid
                                day_selection.loc[:, 'delta_t'] = day_selection['time'].diff()
                                day_selection_min_delta = day_selection['delta_t'].min()
                                day_selection_mode = day_selection['delta_t'].value_counts().index[0]
                                day_selection_max_delta = day_selection['delta_t'].max()
                                day_selection_spread_downward = day_selection_mode - day_selection_min_delta
                                day_selection_spread_upward = day_selection_max_delta - day_selection_mode
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inv_downward'] = day_selection_spread_downward
                                    output_df.at[(year, clean_month, clean_day), 'inv_upward'] = day_selection_spread_upward
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'met_downward'] = day_selection_spread_downward
                                    output_df.at[(year, clean_month, clean_day), 'met_upward'] = day_selection_spread_upward
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unk_downward'] = day_selection_spread_downward
                                    output_df.at[(year, clean_month, clean_day), 'unk_upward'] = day_selection_spread_upward
                            # else leave it as NA.
    # drop irrelevant columns
    output_df = output_df.dropna(axis=1, how='all')
    # drop irrelevant rows
    output_df = output_df.dropna(axis=0, how='all')
    # don't bother with empty dataframes
    if output_df.shape[0] == 0:
        return None
    else:
        return output_df

In [13]:
daily_changes_1200 = daily_delta_t_spread(
    input_dir = '../../../../data_ds_project/testing_yearly_parquet/',
    system_id=1200,
)
daily_changes_1200.head()

inv_downward       inv_upward met_downward met_upward
year month day                                                          
2010 10    3    0 days 00:00:01  0 days 00:30:30         <NA>       <NA>
           4    0 days 00:00:49  0 days 00:00:12         <NA>       <NA>
           5    0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           6    0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           7    0 days 00:00:01  0 days 00:00:01         <NA>       <NA>

In [15]:
daily_changes_1200.describe()

,inv_downward,inv_upward,met_downward,met_upward
count,956,956,2662,2662
unique,28,59,56,62
top,0 days 00:00:01,0 days 00:00:01,0 days 00:00:00,0 days 00:00:00
freq,780,396,1851,1854


In [28]:
daily_changes_1200.iloc[20:40]

inv_downward       inv_upward met_downward met_upward
year month day                                                          
2010 10    23   0 days 00:00:01  0 days 00:15:00         <NA>       <NA>
           24   0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           25   0 days 00:00:01  0 days 00:15:00         <NA>       <NA>
           26   0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           27   0 days 00:00:02  0 days 00:15:00         <NA>       <NA>
           28   0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           29   0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           30   0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           31   0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
     11    1    0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           2    0 days 00:00:01  0 days 01:45:02         <NA>       <NA>
           3    0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           4    0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           5    0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           6    0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           7    0 days 00:00:01  0 days 00:59:59         <NA>       <NA>
           8    0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           9    0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           10   0 days 00:00:00  0 days 00:00:01         <NA>       <NA>
           11   0 days 00:00:59  0 days 00:00:02         <NA>       <NA>

In [41]:
daily_changes_1200.iloc[630:650]

inv_downward       inv_upward met_downward met_upward
year month day                                                          
2012 10    3    0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           4    0 days 00:00:01  0 days 00:05:01         <NA>       <NA>
           5    0 days 00:00:02  0 days 00:00:01         <NA>       <NA>
           6    0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           7    0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           8    0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           9    0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           10   0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           11   0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           12   0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           13   0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           14   0 days 00:00:01  0 days 00:00:03         <NA>       <NA>
           15   0 days 00:00:01  0 days 00:00:02         <NA>       <NA>
           16   0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           23   0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           24   0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           25   0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
           26   0 days 00:00:02  0 days 00:00:01         <NA>       <NA>
           27   0 days 00:00:01  0 days 00:00:01         <NA>       <NA>
     11    1    0 days 00:00:00  0 days 00:00:01         <NA>       <NA>

In [33]:
daily_changes_1200.iloc[930:950]

inv_downward       inv_upward     met_downward  \
year month day                                                      
2013 12    6    0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           7    0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           8    0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           9    0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           10   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           11   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           12   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           13   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           14   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           15   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           16   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           17   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           18   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           19   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           20   0 days 00:02:45  0 days 00:00:00  0 days 00:02:45   
           21   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           22   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           23   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           24   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   
           25   0 days 00:00:00  0 days 00:00:00  0 days 00:00:00   

                     met_upward  
year month day                   
2013 12    6    0 days 00:00:00  
           7    0 days 00:00:00  
           8    0 days 00:00:00  
           9    0 days 00:00:00  
           10   0 days 00:00:00  
           11   0 days 00:00:00  
           12   0 days 00:00:00  
           13   0 days 00:00:00  
           14   0 days 00:00:00  
           15   0 days 00:00:00  
           16   0 days 00:00:00  
           17   0 days 00:00:00  
           18   0 days 00:00:00  
           19   0 days 00:00:00  
           20   0 days 00:00:00  
           21   0 days 00:00:00  
           22   0 days 00:00:00  
           23   0 days 00:00:00  
           24   0 days 00:00:00  
           25   0 days 00:00:00

In [36]:
daily_changes_1200.iloc[2000:2020]

inv_downward inv_upward     met_downward       met_upward
year month day                                                          
2016 11    11          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           12          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           13          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           14          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           15          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           16          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           17          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           18          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           19          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           20          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           21          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           22          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           23          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           24          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           25          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           26          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           27          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           28          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           29          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           30          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00

In [37]:
daily_changes_1200.iloc[-140:-120]

inv_downward inv_upward     met_downward       met_upward
year month day                                                          
2020 3     9           <NA>       <NA>  0 days 00:00:28  0 days 00:00:28
           10          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           11          <NA>       <NA>  0 days 00:00:28  0 days 00:00:28
           12          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           13          <NA>       <NA>  0 days 00:00:29  0 days 00:00:29
           14          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           15          <NA>       <NA>  0 days 00:00:29  0 days 00:00:29
           16          <NA>       <NA>  0 days 00:00:28  0 days 00:00:28
           17          <NA>       <NA>  0 days 00:00:29  0 days 00:00:29
           18          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           19          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           20          <NA>       <NA>  0 days 00:00:31  0 days 00:00:31
           21          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           22          <NA>       <NA>  0 days 00:00:12  0 days 00:00:12
           23          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           24          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           25          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           26          <NA>       <NA>  0 days 00:00:12  0 days 00:00:12
           27          <NA>       <NA>  0 days 00:00:28  0 days 00:00:28
           28          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24

In [17]:
daily_changes_1200['inv_downward'].value_counts()

inv_downward
0 days 00:00:01    780
0 days 00:00:00     50
0 days 00:00:02     38
0 days 00:00:20     22
0 days 00:00:59     18
0 days 00:00:19      9
0 days 00:00:21      7
0 days 00:00:49      5
0 days 00:00:17      5
0 days 00:00:58      3
0 days 00:02:33      2
0 days 00:00:26      1
0 days 00:00:25      1
0 days 00:14:01      1
0 days 00:00:57      1
0 days 00:00:37      1
0 days 00:01:00      1
0 days 00:01:01      1
0 days 00:00:09      1
0 days 00:00:18      1
0 days 00:00:45      1
0 days 00:00:24      1
0 days 00:00:40      1
0 days 00:00:48      1
0 days 00:00:44      1
0 days 00:00:16      1
0 days 00:00:34      1
0 days 00:02:45      1
Name: count, dtype: int64

In [18]:
daily_changes_1200['met_downward'].value_counts()

met_downward
0 days 00:00:00    1851
0 days 00:00:24     333
0 days 00:00:01     221
0 days 00:00:28      36
0 days 00:00:29      29
0 days 00:00:27      18
0 days 00:00:31      17
0 days 00:00:25      14
0 days 00:00:12      13
0 days 00:00:26      12
0 days 00:00:37      10
0 days 00:00:32      10
0 days 00:00:30       9
0 days 00:00:36       8
0 days 00:00:02       6
0 days 00:00:41       5
0 days 00:00:35       5
0 days 00:00:49       4
0 days 00:00:34       3
0 days 00:00:22       3
0 days 00:00:43       3
0 days 00:00:33       3
0 days 00:00:47       3
0 days 00:00:44       3
0 days 00:02:33       2
0 days 00:02:55       2
0 days 00:03:37       2
0 days 00:00:23       2
0 days 00:00:39       2
0 days 00:00:42       2
0 days 00:00:45       2
0 days 00:00:40       2
0 days 00:00:48       2
0 days 00:00:46       2
0 days 00:00:50       2
0 days 00:00:16       1
0 days 00:02:45       1
0 days 00:02:43       1
0 days 00:02:12       1
0 days 00:02:03       1
0 days 00:01:32       1
0 d

In [16]:
daily_changes_1200.tail()

inv_downward inv_upward     met_downward       met_upward
year month day                                                          
2020 7     22          <NA>       <NA>  0 days 00:00:00  0 days 00:00:00
           23          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           24          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24
           25          <NA>       <NA>  0 days 00:00:28  0 days 00:00:28
           26          <NA>       <NA>  0 days 00:00:24  0 days 00:00:24

In [8]:
def daily_delta_t_tracker(
    input_dir: str,
    system_id: int
):
    '''Find all daily changes for all sub-systems.
    
    Parameters:
    ------------
    input_dir: str
        The path to the results of ac_power_parquet_distiller_yearly.py
        Default is data_ds_project/testing_yearly_parquet, 
        but this may change in the final build.
    system_id: int
        The desired system_id.  
    
    Returns:
    ---------
    output_df: pd.DataFrame or None
        If no (good) data, return None
        Else, return a DataFrame with 1-2 columns (from 'inverter', 'meter', 'unknown')
        and indices corresponding to each day with significant data
        (with int data for each of year/month/day) 
    '''
    # initialize data frame
    # know all data is in 1994 - 2023 range
    num_years = 2023-1994 + 1
    num_days = num_years * 12 * 31
    output_df = pd.DataFrame(
        data = np.full((num_days, 3), fill_value=pd.NA),
        index = pd.MultiIndex.from_product(
            [range(1994, 2024), range(1, 13), range(1, 32)]
        ),
        columns = ['inverter', 'meter', 'unknown'],
        dtype='object'
    )
    output_df.index.names = ['year', 'month', 'day']
    # set target
    if input_dir[-1] == '/':
        test_folder_str = f'{input_dir}{system_id}/'
    else:
        test_folder_str = f'{input_dir}/{system_id}/'
    test_folder = Path(test_folder_str)
    # columns invariant among years accessed, so can just grab a year with no data to grab column names
    empty_year_pq = pq.ParquetDataset(
        test_folder,
        filters=[('year', '==', 1990)]
    )
    empty_year_df = empty_year_pq.read().to_pandas()

    # grab column names
    col_names = empty_year_df.columns
    pow_col_names = col_names[col_names.str.contains('ac_power_kW')]
    for year in range(1994, 2024):
        current_year_pq = pq.ParquetDataset(
            test_folder,
            filters=[
                ('year', '==', year)
            ]            
        )
        current_year_df = current_year_pq.read().to_pandas()
        for pow_col_name in pow_col_names:
            pow_selection = current_year_df[['time', pow_col_name]]
            # drop if no data in the column
            pow_selection = pow_selection.dropna(axis=1, how='all')
            if (pow_selection.shape[1] > 1) and (pow_selection.shape[0] > 10):
                pow_selection.loc[:, 'month'] = pow_selection['time'].dt.month
                pow_selection.loc[:, 'day'] = pow_selection['time'].dt.day
                for month in sorted(pow_selection['month'].unique()):
                    month_selection = pow_selection[pow_selection['month'] == month]
                    clean_month = int(month)
                    if (month_selection.shape[0] > 10):
                        for day in sorted(month_selection['day'].unique()):
                            day_selection = month_selection[month_selection['day'] == day]
                            clean_day = int(day)
                            if day_selection.shape[0] > 2:  # need any nonzero data, I'm afraid
                                day_selection.loc[:, 'delta_t'] = day_selection['time'].diff()
                                day_selection_common_delta = day_selection['delta_t'].value_counts().index[0]
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inverter'] = day_selection_common_delta
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'meter'] = day_selection_common_delta
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unknown'] = day_selection_common_delta
                            elif day_selection.shape[0] == 2:  # error message
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inverter'] = timedelta(hours=11, minutes=59)
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'meter'] = timedelta(hours=11, minutes=59)
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unknown'] = timedelta(hours=11, minutes=59)
                            elif day_selection.shape[0] == 1:
                                if 'inv' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'inverter'] = timedelta(hours=23, minutes=59)
                                elif 'met' in pow_col_name:
                                    output_df.at[(year, clean_month, clean_day), 'meter'] = timedelta(hours=23, minutes=59)
                                else:
                                    output_df.at[(year, clean_month, clean_day), 'unknown'] = timedelta(hours=23, minutes=59)
    # drop irrelevant columns
    output_df = output_df.dropna(axis=1, how='all')
    # drop irrelevant rows
    output_df = output_df.dropna(axis=0, how='all')
    # don't bother with empty dataframes
    if output_df.shape[0] == 0:
        return None
    else:
        return output_df

In [9]:
def extract_delta_t_changes(
    input_dir: str,
    system_id: int,
    na_drop_type: str = 'leading_trailing',
):
    '''Extract the list of days with significant data that flank a change in delta_t.
    (Note: this can be a sudden change or an after-5-months-of-no-data change.)
    Parameters:
    ------------
    input_dir: str
        The path to the results of ac_power_parquet_distiller_yearly.py
        Default is data_ds_project/testing_yearly_parquet/, 
        but this may change in the final build.
    system_id: int
        The desired system_id.
    na_drop_type: str,
        If na_drop_type == "all", will drop all na terms
        If na_drop_type == "leading_trailing", will retain interior na's
        and drop those outside that.
    
    Returns:
    ---------
    out_list: list[pandas.Series]
       A list with one element if one power-attribute,
       and two elements if two power-attributes.
       Each term is a pandas Series object with the dates and common-delta-t's
       flanking a change in the delta-t.
       Again, the dates are in an awkward year/month/day index format
       and not.
    '''
    out_list = []
    my_daily_terms = daily_delta_t_tracker(input_dir=input_dir,
        system_id=system_id
    )
    for col in my_daily_terms.columns:
        col_forwards = f'{col}_forward_difference'
        col_backwards = f'{col}_backward_difference'
        my_daily_terms.loc[:, col_forwards] = -my_daily_terms[col].diff(periods=-1)
        my_daily_terms.loc[:, col_backwards] = my_daily_terms[col].diff(periods=1)
        # grab relevant columns and drop_na
        my_col_relevant = my_daily_terms[[col, col_forwards, col_backwards]].copy(deep=True)
        # first and last rows will always have NaT terms, so grab NaT rows
        # and any terms with large differences
        my_col_relevant = my_daily_terms[
            my_daily_terms[col_forwards].isna()
            | my_daily_terms[col_backwards].isna()
            | ((my_daily_terms[col_forwards] - my_daily_terms[col_backwards]) > timedelta(seconds=15))
            | ((my_daily_terms[col_backwards] - my_daily_terms[col_forwards]) > timedelta(seconds=15))
        ]
        return_column = my_col_relevant[col]
        # now drop NA's as required
        if na_drop_type == 'all':
            return_column = return_column.dropna()
        elif na_drop_type == 'leading_trailing':
            first_ind = return_column.first_valid_index()
            last_ind = return_column.last_valid_index()
            return_column = return_column.loc[first_ind:last_ind]
        out_list.append(return_column)
    return out_list

In [10]:
for system_id in enough_data_parquet_power_list:
    my_list = extract_delta_t_changes('../../../../data_ds_project/testing_yearly_parquet',
                                      system_id,
                                      na_drop_type='all')
    print(f'System {system_id}')
    if len(my_list) == 2:
        print('first group')
        print(my_list[0])
        print('')
        print('second group')
        print(my_list[1])
        print('')
    else:
        print(my_list[0])

System 4
year  month  day
2007  8      26     0 days 00:15:00
2008  4      3      0 days 00:15:00
             4      0 days 07:00:00
             5      0 days 00:30:00
             6      0 days 00:15:00
2009  6      3      0 days 00:15:00
2010  1      22     0 days 00:01:00
2023  2      28     0 days 00:01:00
Name: unknown, dtype: object
System 10
year  month  day
2000  5      3             23:59:00
             10     0 days 00:01:00
2006  1      9      0 days 00:15:00
2009  4      21     0 days 00:15:00
             22            23:59:00
                         ...       
2019  3      16     0 days 02:07:00
             17     0 days 02:50:00
             18     0 days 00:08:00
             19     0 days 00:01:00
2023  2      28     0 days 00:01:00
Name: unknown, Length: 96, dtype: object
System 33
year  month  day
2010  11     8      0 days 00:01:00
2019  2      25     0 days 00:01:00
             26            23:59:00
      3      21            23:59:00
             22     0 

System 1368 is noisy.  What is going on there?

In [21]:
all_daily_delta_ts_1368 = daily_delta_t_tracker(input_dir='../../../../data_ds_project/testing_yearly_parquet',
        system_id=1368
)

In [22]:
all_daily_delta_ts_1368.value_counts()

inverter       
0 days 00:15:02    1815
0 days 00:15:03     137
0 days 00:15:01      82
0 days 00:15:04       3
0 days 00:00:01       2
0 days 00:14:55       1
0 days 00:14:57       1
0 days 00:14:59       1
0 days 00:14:54       1
Name: count, dtype: int64

In [23]:
daily_spreads_1368 = daily_delta_t_spread(input_dir='../../../../data_ds_project/testing_yearly_parquet',
        system_id=1368
)

In [26]:
daily_spreads_1368.iloc[20:40]

inv_downward       inv_upward
year month day                                  
2013 6     18   0 days 00:00:10  0 days 00:00:03
           19   0 days 00:00:09  0 days 00:00:03
           20   0 days 00:00:10  0 days 00:00:02
           21   0 days 00:00:10  0 days 00:00:02
           22   0 days 00:00:09  0 days 00:00:03
           23   0 days 00:00:11  0 days 00:00:01
           24   0 days 00:00:10  0 days 00:00:03
           25   0 days 00:00:10  0 days 00:00:03
           26   0 days 00:00:10  0 days 00:00:03
           27   0 days 00:00:09  0 days 00:00:02
           28   0 days 00:00:10  0 days 00:00:02
           29   0 days 00:00:10  0 days 00:00:02
           30   0 days 00:00:10  0 days 00:00:03
     7     1    0 days 00:00:10  0 days 00:00:03
           2    0 days 00:00:09  0 days 00:00:03
           3    0 days 00:00:10  0 days 00:00:02
           4    0 days 00:00:10  0 days 00:00:02
           5    0 days 00:00:10  0 days 00:00:03
           6    0 days 00:00:09  0 days 00:00:02
           7    0 days 00:00:10  0 days 00:00:02

In [27]:
daily_spreads_1368.iloc[-40:-20]

inv_downward       inv_upward
year month day                                  
2018 11    22   0 days 00:15:01  0 days 04:44:53
           23   0 days 00:15:01  0 days 00:04:10
           24   0 days 00:15:01  0 days 00:05:03
           25   0 days 00:15:01  0 days 05:17:24
           26   0 days 00:15:01  0 days 00:09:03
           27   0 days 00:15:01  0 days 00:06:33
           28   0 days 00:15:01  0 days 00:07:19
           29   0 days 00:15:02  0 days 06:14:57
           30   0 days 00:15:01  0 days 00:05:36
     12    1    0 days 00:15:01  0 days 05:44:59
           2    0 days 00:15:01  0 days 00:11:12
           3    0 days 00:15:01  0 days 00:04:32
           4    0 days 00:15:01  0 days 00:08:40
           5    0 days 00:15:00  0 days 00:06:07
           6    0 days 00:15:01  0 days 00:06:52
           7    0 days 00:15:01  0 days 02:45:01
           8    0 days 00:15:01  0 days 00:08:24
           9    0 days 00:15:01  0 days 00:09:07
           10   0 days 00:15:01  0 days 00:09:15
           11   0 days 00:15:01  0 days 03:29:57